In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [2]:
!curl -o hw_data.csv $data

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0  2163k      0 --:--:-- --:--:-- --:--:-- 2177k


In [3]:
df = pd.read_csv('hw_data.csv')

In [9]:
categorical_cols = list(df.select_dtypes(include=['object', 'string']).columns)
numerical_cols = list(df.select_dtypes(include=['number']).columns)

numerical_cols = [c for c in numerical_cols if c != 'converted']
categorical_cols = [c for c in categorical_cols if c != 'converted']

df[categorical_cols] = df[categorical_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

### Q1. ROC AUC feature importance

In [13]:
auc_scores = {}

for feature in numerical_cols:
    score = df_train[feature]
    auc = roc_auc_score(y_train, score)
    if auc < 0.5:
        auc = roc_auc_score(y_train, -score)
    auc_scores[feature] = auc
    print(f"Feature: {feature:25} AUC: {auc:.3f}")

best_feature = max(auc_scores, key=auc_scores.get)
print(f"\nBest AUC: {best_feature}\n")

Feature: annual_income             AUC: 0.608
Feature: number_of_courses_viewed  AUC: 0.723
Feature: interaction_count         AUC: 0.765
Feature: lead_score                AUC: 0.788

Best AUC: lead_score



### Q2. What's the AUC of this model on the validation dataset?

In [15]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, y_pred_proba)

print(f"AUC: {auc:.3f}\n")

AUC: 0.732



### Q3 & Q4. Precision, Recall, and F1 Score Across Thresholds

In [16]:
thresholds = np.linspace(0.0, 1.0, 101)
precisions, recalls, f1_scores = [], [], []

for t in thresholds:
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)
    
    predict_positive = (y_pred_proba >= t)
    predict_negative = (y_pred_proba < t)
    
    tp = (predict_positive & actual_positive).sum()
    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    
    precisions.append(p)
    recalls.append(r)
    f1_scores.append(f1)

diffs = [abs(p - r) for p, r in zip(precisions, recalls)]
# Mask cases where both are zero
for i, (p, r) in enumerate(zip(precisions, recalls)):
    if p == 0 and r == 0:
        diffs[i] = float('inf')

intersection_threshold = thresholds[np.argmin(diffs)]
print(f"P/R Intersection Threshold: {intersection_threshold:.2f}")

best_f1_idx = np.argmax(f1_scores)
best_f1_threshold = thresholds[best_f1_idx]
print(f"\nMax F1 Threshold: {best_f1_threshold:.2f}\n")

P/R Intersection Threshold: 0.63

Max F1 Threshold: 0.41



### Q5. How large is standard deviation of the scores across different 5-folds?

In [17]:
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical_cols + numerical_cols].to_dict(orient='records')
    
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)
    
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X_train, y_train)
    
    return dv, model

def predict (df, dv, model):
    dicts = df[categorical_cols + numerical_cols].to_dict(orient='records')
    
    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]
    
    return y_pred

In [20]:
y_full_train = df_full_train['converted'].values
df_full_train_features = df_full_train[categorical_cols + numerical_cols].reset_index(drop=True)

n_splits = 5

kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train_features):
    df_train = df_full_train_features.iloc[train_idx]
    df_val = df_full_train_features.iloc[val_idx]

    y_train = y_full_train[train_idx]
    y_val = y_full_train[val_idx]
    
    dv, model = train(df_train, y_train, C=1.0)
    y_pred = predict(df_val, dv, model)
    
    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

print('C=%s %.3f +- %.3f' % (1.0, np.mean(scores), np.std(scores)))

C=1.0 0.732 +- 0.007


### Q6. Which C leads to the best mean ROC AUC?

In [21]:
for C in [0.000001, 0.001, 1]:
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)
    scores = []

    for train_idx, val_idx in kfold.split(df_full_train_features):
        df_train = df_full_train_features.iloc[train_idx]
        df_val = df_full_train_features.iloc[val_idx]

        y_train = y_full_train[train_idx]
        y_val = y_full_train[val_idx]
        
        dv, model = train(df_train, y_train, C=C)
        y_pred = predict(df_val, dv, model)
        
        auc = roc_auc_score(y_val, y_pred)
        scores.append(auc)

    print('C=%-8s | Mean AUC: %.3f | Std: %.3f' % (C, np.mean(scores), np.std(scores)))

C=1e-06    | Mean AUC: 0.617 | Std: 0.019
C=0.001    | Mean AUC: 0.749 | Std: 0.010
C=1        | Mean AUC: 0.732 | Std: 0.007
